# Custom Layers and Functions

Our concise models so far have been assembled from the framework's library
of layers: fully connected layers, activation functions, dropout. Research
moves faster than libraries, however, and a layer that a paper proposes can be
missing from them for years. RMSNorm, a layer that rescales each activation
vector to a fixed size, is an example. It was published in 2019
[@Zhang.Sennrich.2019], and PyTorch added it to its library of layers
in 2024. Until then, each PyTorch codebase that used it carried its own
implementation or borrowed one from another package. How do we write such a
layer ourselves, so that it behaves like a built-in one? Its parameters must
reach the optimizer, its state must be saved and moved to the GPU, and it
must fit into containers next to other layers.

Sometimes the problem is the derivative instead. Rounding, which is needed
to store weights as 4-bit integers, has derivative zero almost everywhere,
so a weight that is rounded in the forward pass receives no gradient and
never learns.

This section writes both kinds of extension. A custom layer is a subclass
of the module class of that section. Recall that a
module owns registered parameters and child modules and defines a forward
computation, and that everything the framework does to a whole model, from
collecting parameters for the optimizer to saving a checkpoint, walks the
registered state. We write a layer without state, then RMSNorm with a
learned parameter, then a layer with fixed state that must be saved but
never trained. Finally we replace the derivative of an operation with a
rule of our own, which lets gradients pass through rounding.

In [1]:
import math
import tensorflow as tf

## Layers without Parameters

The smallest custom layer has no state at all. `CenteredLayer` subtracts the
mean from its input. It inherits from the module class and supplies only the
forward method; there is nothing to set up.

In [2]:
class CenteredLayer(tf.keras.layers.Layer):
    def __init__(self):
        super().__init__()

    def call(self, X):
        return X - tf.reduce_mean(X)

Feeding data through confirms that it does what it says:

In [3]:
layer = CenteredLayer()
layer(tf.constant([1.0, 2, 3, 4, 5]))

<tf.Tensor: shape=(5,), dtype=float32, numpy=array([-2., -1.,  0.,  1.,  2.], dtype=float32)>

The class has the same interface as a built-in layer, so a sequential
container can call it like any other child. Placed after a linear layer, it
centers the linear layer's outputs. Their mean should be zero; since the
mean is computed in floating-point arithmetic, we see a number of size
$10^{-8}$ or smaller rather than exactly zero, which is rounding error, not a bug.

In [4]:
net = tf.keras.Sequential([tf.keras.layers.Dense(128), CenteredLayer()])
Y = net(tf.random.uniform((4, 8)))
tf.reduce_mean(Y)

<tf.Tensor: shape=(), dtype=float32, numpy=0.0>

## Layers with Parameters

In that section we saw that the scale of the
activations can grow or shrink from one layer to the next, by a factor that
depends on the weights. A *normalization layer* removes this dependence: it
rescales each input vector to a fixed size and lets a learned parameter set
the size instead. We implement the simplest one, *RMSNorm*.

### RMSNorm

The root mean square of a vector $\mathbf{x} \in \mathbb{R}^d$ is
$\sqrt{\frac{1}{d}\sum_{i=1}^d x_i^2}$. RMSNorm divides each example's
vector by it and multiplies the result elementwise by a learned *gain*
$\mathbf{g} \in \mathbb{R}^d$:

$$
\textrm{RMSNorm}(\mathbf{x}) = \frac{\mathbf{x}}{\sqrt{\frac{1}{d} \sum_{i=1}^{d} x_i^2 + \epsilon}} \odot \mathbf{g}.
$$

The small constant $\epsilon$, such as $10^{-6}$, keeps the denominator away
from zero. With the gain at its initial value $\mathbf{g} = \mathbf{1}$, the
output has root mean square close to 1, whatever the scale of the input.
For $\mathbf{x} = (3, 4)$ the mean square is $(9 + 16)/2 = 12.5$, the root
mean square is $3.536$, and the output is $(0.849, 1.131)$, whose mean
square is 1. The input $(300, 400)$ gives the same output.
the figure shows the geometry: RMSNorm keeps the direction of
$\mathbf{x}$, discards its length by moving it onto the sphere of radius
$\sqrt{d}$, and the gain then stretches each coordinate.

![RMSNorm in two dimensions. Left: with gain 1, every input (blue) moves along its ray from the origin onto the circle of radius $\sqrt{2}$ (orange); two inputs on the same ray, at lengths 0.6 and 2.9, land on the same point. Right: the gain $\mathbf{g} = (1.5, 0.5)$ stretches the circle into an ellipse with semi-axes $1.5\sqrt{2}$ and $0.5\sqrt{2}$.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-rmsnorm.svg)

RMSNorm simplifies *layer normalization* [@Ba.Kiros.Hinton.2016],
which first subtracts the mean of the entries of each example's vector,
as a per-example `CenteredLayer` would, and then adds a learned shift after
the gain. Zhang and Sennrich found that the centering contributes little to
training, and many recent language models use RMSNorm, among them Llama 3
8B [@Grattafiori.Dubey.Jauhri.ea.2024], the 8B model of
that section.

A layer with something to learn must register its parameters. In Keras,
`add_weight` creates and registers a variable (that section).
Keras creates variables in the method `build`, which runs on the first call
once the input shape is known, so our layer need not be told its width.

In [5]:
class RMSNorm(tf.keras.layers.Layer):
    def __init__(self, eps=1e-6):
        super().__init__()
        self.eps = eps

    def build(self, X_shape):
        self.gain = self.add_weight(name='gain', shape=[X_shape[-1]],
                                    initializer='ones')

    def call(self, X):
        rms = tf.sqrt(
            tf.reduce_mean(X ** 2, axis=-1, keepdims=True) + self.eps)
        return self.gain * X / rms

The worked example, and the same vector scaled by 100, both give
$(0.849, 1.131)$:

In [6]:
norm = RMSNorm()
norm(tf.constant([[3.0, 4.0], [300.0, 400.0]]))

<tf.Tensor: shape=(2, 2), dtype=float32, numpy=
array([[0.84852815, 1.1313709 ],
       [0.84852815, 1.1313709 ]], dtype=float32)>

### Registration

We wrote RMSNorm as a module, not as a function with a gain tensor kept
beside it, because registration makes a custom layer behave like a built-in
one in four ways: its parameters are found, it works inside containers, its
state is saved and loaded, and it moves between devices. We check each
once.

First, the gain was registered when `add_weight` ran inside `build`. An
optimizer given `norm.trainable_variables` finds and updates it.

In [7]:
norm.trainable_variables

[<Variable path=rms_norm/gain, shape=(2,), dtype=float32, value=[1. 1.]>]

Second, the layer works inside a sequential container, between two linear
layers:

In [8]:
net = tf.keras.Sequential([tf.keras.layers.Dense(8), RMSNorm(),
                           tf.keras.layers.Dense(2)])
net(tf.random.normal((4, 20))).shape

TensorShape([4, 2])

Third, its state is saved and loaded with everything else. We copy the
state of `net` into a second model with the same structure and its own
random initial weights; afterwards the two compute the same outputs
(that section writes such state to disk).

`get_weights` returns the model's weights, the gain included, as a list of
arrays, and `set_weights` loads the list into the clone once a first call
has built it.

In [9]:
clone = tf.keras.Sequential([tf.keras.layers.Dense(8), RMSNorm(),
                             tf.keras.layers.Dense(2)])
X = tf.random.normal((4, 20))
clone(X)  # A first call builds the clone so its weights exist
clone.set_weights(net.get_weights())
bool(tf.reduce_all(net(X) == clone(X)))

True

Fourth, it moves, although Keras places variables when it creates them
instead of moving them later. `add_weight` puts the gain on the default
device, a GPU if one is visible, and an operation runs where its inputs
are; a `tf.device` scope at construction chooses another device
(that section). There is no call to make here: the gain was
created through `add_weight`, so it is placed like any built-in weight.

None of these four checks required code in `RMSNorm` beyond the
registration of its gain. Optimizers, containers, checkpoints, and device
transfers handle the custom layer through the same interface as a built-in
one.

### Comparison with Framework Implementations

Where the library provides the same layer, comparing the outputs of the two
implementations tests ours.

Keras's `RMSNormalization` calls its gain `scale` and its divisor guard
`epsilon`. We give both implementations the same gain, other than all ones,
and compare their outputs; once both layers are built, `get_weights` and
`set_weights` copy the gain across.

In [10]:
ours, native = RMSNorm(), tf.keras.layers.RMSNormalization(epsilon=1e-6)
X = tf.random.normal((16, 8))
ours(X)  # Build both so the weights exist
native.build(X.shape)
ours.gain.assign(tf.linspace(0.5, 1.5, 8))
native.set_weights(ours.get_weights())
bool(tf.experimental.numpy.allclose(ours(X), native(X)))

True

## Layers with Fixed State

Some layers hold arrays that are neither learned nor recomputed: they are
fixed when the layer is built and must stay the same for the life of the
model. A simple example is a layer of *random features*
[@Rahimi.Recht.2007]. It multiplies a minibatch $\mathbf{X}$ by a
random matrix $\mathbf{W} \in \mathbb{R}^{d \times D}$ with standard normal
entries, adds a random offset $\mathbf{b} \in \mathbb{R}^D$ with entries
uniform on $[0, 2\pi]$, and applies the cosine elementwise:

$$\mathbf{Z} = \sqrt{2/D}\, \cos(\mathbf{X}\mathbf{W} + \mathbf{b}).$$

Each row of $\mathbf{Z}$ holds $D$ nonlinear features of the corresponding
example, and a linear layer on top of them can fit nonlinear functions of
the input while only the linear layer is trained.

$\mathbf{W}$ and $\mathbf{b}$ are drawn once and must then not change.
They are not parameters: the optimizer must not update them. Yet they must
be saved with the model. A model rebuilt from a checkpoint that lacked them
would draw a new $\mathbf{W}$, so the trained linear layer on top would
receive different features and its outputs would be meaningless. They must
also move to the GPU with the rest of the model. that section
called such arrays *buffers*: state that is saved and moved with the model
but receives no gradient. Custom layers are where buffers are created.

In Keras a buffer is a variable created with `add_weight(trainable=False)`:
it is among the layer's weights, so `get_weights` saves it, but not among
its trainable variables, so no optimizer updates it. Each `add_weight` gets
an initializer of its own, because one Keras initializer object returns the
same values every time it is called.

In [11]:
class RandomFeatures(tf.keras.layers.Layer):
    def __init__(self, d, num_features):
        super().__init__()
        normal = tf.keras.initializers.RandomNormal(stddev=1.0)
        uniform = tf.keras.initializers.RandomUniform(0, 2 * math.pi)
        self.W = self.add_weight(name='W', shape=(d, num_features),
                                 initializer=normal, trainable=False)
        self.b = self.add_weight(name='b', shape=(num_features,),
                                 initializer=uniform, trainable=False)

    def call(self, X):
        return math.sqrt(2 / self.b.shape[0]) * tf.cos(X @ self.W + self.b)

Two layers of the same shape draw different matrices, so their features
differ. Copying the state of one into the other copies $\mathbf{W}$ and
$\mathbf{b}$, after which the two agree. The layer has no parameters, and
its state consists of the two buffers:

In [12]:
feats, fresh = RandomFeatures(20, 64), RandomFeatures(20, 64)
X = tf.random.normal((4, 20))
before = bool(tf.reduce_all(feats(X) == fresh(X)))
fresh.set_weights(feats.get_weights())
print(feats.trainable_weights, [w.name for w in feats.weights])
print(before, bool(tf.reduce_all(feats(X) == fresh(X))))

[] ['W', 'b']
False True


The trainable weights are empty, and the weights are `W` and `b`. Stored
as plain tensor attributes, they would be missing from `get_weights`, and
the loaded layer would keep its own random matrix.

The same mechanism holds the fixed tables and running averages of layers in
later chapters, such as the running statistics of batch normalization in
that section.

## Functions with Custom Gradients

So far we wrote forward computations and let automatic differentiation
derive the backward pass. It differentiates exactly the operations we ran,
and occasionally that is the problem. A model trained to work with weights
stored as 4-bit integers, which take a quarter of the memory of 16-bit
floating-point numbers, rounds its weights in the forward pass. Rounding is
constant between integers, so its derivative is zero almost everywhere, and
any weight behind a rounding operation receives a zero gradient:

In [13]:
w = tf.Variable([0.9, 1.4, 2.6])
with tf.GradientTape() as tape:
    y = tf.reduce_sum(tf.round(w))
tape.gradient(y, w, unconnected_gradients=tf.UnconnectedGradients.ZERO)

<tf.Tensor: shape=(3,), dtype=float32, numpy=array([0., 0., 0.], dtype=float32)>

TensorFlow declares rounding non-differentiable, so without the argument
`unconnected_gradients` the tape returns `None`; the argument asks for the
zeros that calculus gives.

The gradient is zero, as calculus says, and useless for training. The
*straight-through estimator* [@Bengio.Leonard.Courville.2013] replaces
it: the forward pass still rounds, and the backward pass treats the
rounding as if it were the identity, whose derivative is 1.
the figure shows the two passes. Automatic differentiation cannot
produce this rule, because it is not the derivative of what the forward
pass computes, so we supply the backward pass ourselves.

![The straight-through estimator. Forward: the staircase round(x) stays close to the identity (dashed). Backward: the true derivative of rounding is zero almost everywhere; the estimator uses the derivative of the identity, 1, instead.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-ste.svg)

In TensorFlow the tool is the decorator `@tf.custom_gradient`: the
decorated function returns its output together with its backward rule:

In [14]:
@tf.custom_gradient
def round_ste(X):
    def grad(upstream):
        # Treat the forward pass as the identity: pass the gradient on
        return upstream
    return tf.round(X), grad

Two rules govern the decorator. First, the decorated function returns a
pair: the output and the gradient function. The gradient function is
defined inside the forward pass, so it can use any value that the forward
pass computed, such as the input. Second, `grad` receives the gradient of
the loss with respect to the output and returns one gradient per argument
of the decorated function. A function that reads a `tf.Variable` must also
return gradients for the variables, through a keyword argument `variables`
of `grad`; TensorFlow raises an error that names this requirement if it is
missing.

To check the rule, we multiply the rounded values by known weights
$(1, 2, 3)$ and sum. The straight-through estimator should deliver exactly
these weights as the gradient at $\mathbf{w}$:

In [15]:
with tf.GradientTape() as tape:
    y = tf.reduce_sum(round_ste(w) * tf.constant([1.0, 2.0, 3.0]))
tape.gradient(y, w)

<tf.Tensor: shape=(3,), dtype=float32, numpy=array([1., 2., 3.], dtype=float32)>

The gradient arrives at $\mathbf{w}$ unchanged, as if the rounding were the
identity, while the forward pass computed with the rounded values. Other
surrogates are possible. A quantizer to 4-bit integers also clips its input
to the representable range, $-8$ to $7$, and a common variant of the
estimator passes the gradient only inside that range and sets it to zero
outside, where the clipped output does not change. The surrogate is a
modeling choice, and the right one depends on the quantizer it stands in
for.

For this particular rule there is also a shortcut without a hand-written
backward pass. A *stop-gradient* operation is the identity in the forward
pass and has derivative zero in the backward pass, so
$X + \textrm{stop}(\textrm{round}(X) - X)$ computes $\textrm{round}(X)$ in
the forward pass while the backward pass sees only the leading $X$. The
general mechanism is needed when the surrogate is not the identity, such as
the clipping rule above.

In TensorFlow the stop-gradient operation is `tf.stop_gradient`. The two
definitions agree in value and in gradient:

In [16]:
def round_ste2(X):
    return X + tf.stop_gradient(tf.round(X) - X)

def grad_of(f):
    with tf.GradientTape() as tape:
        y = tf.reduce_sum(f(w) * tf.constant([1.0, 2.0, 3.0]))
    return tape.gradient(y, w)

allclose = tf.experimental.numpy.allclose
(bool(allclose(round_ste(w), round_ste2(w))),
 bool(allclose(grad_of(round_ste), grad_of(round_ste2))))

(True, True)

A custom gradient changes the derivative of a computation built from
existing tensor operations; it does not create new operations. Writing new
*kernels*, programs that run on the accelerator itself, is a separate craft
that this book does not cover. that section shows how to get
performance out of the operations that the libraries already provide.

We can now add to a model any layer that tensor operations can express,
with parameters, buffers, or a backward rule of our own, and have it
trained, saved, and moved like a built-in one. What we have not asked is
whether its arithmetic is accurate: RMSNorm squares its inputs, and the sum
of squares can overflow in low precision. that section takes up the
number formats themselves.

## Summary

A custom layer is a subclass of the module class with a forward method.
Registration does the rest: a learned parameter registered with the module
is found by the optimizer, saved, loaded, and moved like the parameters of
built-in layers, as we checked for the gain of RMSNorm. State that is fixed,
such as the random matrix of a random-feature layer, is registered as a
buffer, which is saved and moved but never trained. When the derivative of
an operation is useless, as for rounding, a custom backward rule such as
the straight-through estimator replaces it. A library implementation is
preferable when it computes the same thing; otherwise the custom layer,
tested against its definition, is the implementation.

In Keras, `add_weight` creates learned state, `add_weight(trainable=False)`
creates buffers, and `@tf.custom_gradient` lets a function return its own
backward rule; `tf.stop_gradient` covers the straight-through case in one
expression.

## Exercises

1. **Centering over the batch.** `CenteredLayer` subtracts the mean over
   every entry of its input, the batch axis included. Consider the network
   of this section, a linear layer with bias $\mathbf{b}$ followed by
   `CenteredLayer`.
    1. Show that the output for one example depends on the other examples
       in the minibatch. State when evaluating the network on one example
       at a time gives the same outputs as evaluating it on the whole
       minibatch at once.
    1. Show that adding the same constant to every entry of $\mathbf{b}$
       leaves the output unchanged, and conclude which direction in the
       space of biases receives zero gradient.
    1. A per-example variant subtracts from each row its own mean over the
       feature axis, as layer normalization does. State which of the two
       properties above this variant keeps.

2. [code] **Scale invariance of RMSNorm.** ● Set $\epsilon = 0$.
    1. Show that $\textrm{RMSNorm}(c\mathbf{x}) = \textrm{RMSNorm}(\mathbf{x})$
       for every $c > 0$.
    1. Differentiate both sides with respect to $c$ at $c = 1$ and conclude
       that the Jacobian $\mathbf{J}$ of RMSNorm at $\mathbf{x}$ satisfies
       $\mathbf{J}\mathbf{x} = \mathbf{0}$.
    1. Conclude that the gradient of any loss with respect to the input
       $\mathbf{x}$ of RMSNorm is orthogonal to $\mathbf{x}$, and check
       this with automatic differentiation for a random $\mathbf{x}$ and a
       random linear loss. What does a small gradient step do to the
       length of $\mathbf{x}$?

3. [code] **RMSNorm bias.** Add an optional learned bias $\mathbf{b}$ to
   `RMSNorm` (a shift applied after the gain, restoring part of what layer
   normalization had), and verify that `get_weights` on a model containing
   it grows by the expected entry. Then place the layer directly before a
   linear layer with weight $\mathbf{W}$ and bias $\mathbf{c}$, and show
   that $\mathbf{b}$ can be folded into $\mathbf{c}$ without changing the
   outputs. What is the folded bias?

4. [code] **Random features approximate a kernel.** Let $\mathbf{w}$ have
   independent standard normal entries and $b$ be uniform on $[0, 2\pi]$.
    1. Using
       $\cos\alpha\cos\beta = \frac{1}{2}(\cos(\alpha - \beta) + \cos(\alpha + \beta))$
       and $\mathbb{E}[\cos(\mathbf{w}^\top\mathbf{u})] = \exp(-\|\mathbf{u}\|^2/2)$,
       show that the features $\mathbf{z}(\mathbf{x})$ of `RandomFeatures`
       satisfy
       $\mathbb{E}[\mathbf{z}(\mathbf{x})^\top \mathbf{z}(\mathbf{x}')] = \exp(-\|\mathbf{x} - \mathbf{x}'\|^2/2)$
       for every number $D$ of features.
    1. Check this numerically for two inputs at distance 1, with $D = 10$,
       100, 1,000, and 10,000, averaging the error over several layers for
       each $D$. How does the error shrink as $D$ grows?
    1. Train a linear layer on top of `RandomFeatures` to fit
       $y = \sin(3x)$ for $x$ uniform on $[-2, 2]$, and compare with a
       linear model on $x$ alone.

    *Adapted from [@Rahimi.Recht.2007].*

5. [code] **Custom gradients.**
    1. Implement a clamp with a custom gradient: a `@tf.custom_gradient`
       function whose forward is `tf.clip_by_value(X, lo, hi)` and whose
       backward passes the gradient only where the input lay strictly
       inside the clamp range; the gradient closure must capture `X`.
       Compare its gradients with those of the native `tf.clip_by_value`
       for inputs inside, outside, and exactly on the boundary. Which
       convention does the native operation use at the boundary?
    1. Implement a second custom operation, GELU or Softplus, with a
       hand-derived backward. Check it numerically against the automatic
       derivative of the equivalent built-in expression and report the
       largest discrepancy.

    *Adapted from the PyTorch tutorial
    [Learning PyTorch with Examples](https://docs.pytorch.org/tutorials/beginner/pytorch_with_examples.html)
    and the JAX notebook
    [Custom derivative rules](https://docs.jax.dev/en/latest/notebooks/Custom_derivative_rules_for_Python_code.html).*

6. [extended] **Bucketing with a straight-through estimator.** ●
   the figure passes a surrogate gradient through `round(x)`.
   Extend the idea to a layer whose forward pass maps a score $s$ to the
   bucket index $\lfloor K\sigma(s) \rfloor$ and whose backward pass treats
   the floor as the identity. Return the index as a floating-point tensor so
   that gradients can reach $s$. Place the layer with $K = 4$ between two
   linear layers of output width one, and train the network on inputs
   $\mathbf{x} \in \mathbb{R}^{10}$ with targets
   $y = \lfloor 4\sigma(\mathbf{w}^\top \mathbf{x}) \rfloor$ for a fixed
   random $\mathbf{w}$. Compare the final training loss with that of the
   same network in which the floor keeps its true gradient, and explain
   which parameters learn in each case.

[Discussions](https://d2l.discourse.group/t/36755)